In [1]:
# ── Cell 1: Environment Check ──────────────────────────────────────────────────
import subprocess, torch, sys

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU         : {props.name}")
    print(f"VRAM        : {props.total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ Running on CPU - might be slower but Kokoro is very efficient!")

Python      : 3.12.13
PyTorch     : 2.10.0+cu128
CUDA        : True
GPU         : Tesla T4
VRAM        : 14.6 GB


In [ ]:
# ── Cell 2: Install EdgeTTS (Best Quality & Speed) ─────────────────────────────
import subprocess, sys

def run_pip(*args):
    """Run pip install."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print(f"$ {' '.join(cmd)}")
    subprocess.check_call(cmd)

print("📦 Installing EdgeTTS (Microsoft Neural Voices)...")
# edge-tts: high quality, fast, no GPU needed
run_pip("edge-tts", "fastapi", "uvicorn", "soundfile", "numpy", "pyngrok", "requests")


print("✅ Dependencies installed.")

print("\n🔄 Restarting kernel...")
from IPython import get_ipython as _gip
_gip().kernel.do_shutdown(restart=True)

📦 Installing EdgeTTS (Microsoft Neural Voices)...
$ /usr/bin/python3 -m pip install edge-tts fastapi uvicorn soundfile numpy pyngrok requests
✅ Dependencies installed.

🔄 Restarting kernel...


{'status': 'ok', 'restart': True}

: 

In [1]:
# ── Cell 3: Verify EdgeTTS Configuration ───────────────────────────────────────
import edge_tts
import asyncio

# List available voices to confirm
async def list_voices():
    voices = await edge_tts.VoicesManager.create()
    english = [v for v in await voices.list() if "en-US" in v['ShortName']]
    print(f"Found {len(english)} English voices.")
    # Recommend a good default
    print("Recommended for Interviewer: 'en-US-JennyNeural' (Female) or 'en-US-GuyNeural' (Male)")
    return english

print("✅ EdgeTTS ready (No heavy model loading needed!)")
print("   This uses Microsoft's cloud neural voices (requires internet).")
print("   Quality: ⭐⭐⭐⭐⭐ (Indistinguishable from human)")
print("   Speed:   🚀 Fast")

✅ EdgeTTS ready (No heavy model loading needed!)
   This uses Microsoft's cloud neural voices (requires internet).
   Quality: ⭐⭐⭐⭐⭐ (Indistinguishable from human)
   Speed:   🚀 Fast


In [2]:
# ── Cell 5: FastAPI TTS Server (High Quality & Auto-Discovery) ────────────────
import os
import io
import asyncio
import edge_tts
import uvicorn
from fastapi import FastAPI, Request, HTTPException, Header
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import threading
import secrets
import socket
import re
import sys

# ── Config ────────────────────────────────────────────────────────────────────
VOICE = "en-US-JennyNeural" # 'en-US-JennyNeural' (Female) or 'en-US-GuyNeural' (Male)
RATE = "+0%"
PITCH = "+0Hz"

# ── Utils ─────────────────────────────────────────────────────────────────────
def find_available_port(start_port=8080, max_attempts=10):
    for port in range(start_port, start_port + max_attempts):
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.bind(("0.0.0.0", port)) # CHECK 0.0.0.0 to ensure availability
            sock.close()
            return port
        except OSError:
            continue
    return start_port

HOST      = "0.0.0.0" # Listen on all interfaces (fixes localhost vs 127.0.0.1 issues)
PORT      = find_available_port(8080)
API_KEY   = secrets.token_urlsafe(24)

app = FastAPI(title="EdgeTTS Server", version="2.1")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok", "engine": "EdgeTTS (Microsoft Neural)", "voice": VOICE}

@app.post("/v1/tts")
async def tts(request: Request):
    body = await request.json()
    text = (body.get("text") or "").strip()
    if not text: raise HTTPException(400, "Missing text")
    text = text.replace("[S1]", "").replace("[S2]", "").strip()

    try:
        communicate = edge_tts.Communicate(text, VOICE, rate=RATE, pitch=PITCH)
        audio_data = b""
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                audio_data += chunk["data"]
        return Response(content=audio_data, media_type="audio/mpeg")
    except Exception as e:
        print(f"TTS Error: {e}")
        raise HTTPException(500, str(e))

def _run():
    # uvicorn.run blocks, so run in thread
    config = uvicorn.Config(app, host=HOST, port=PORT, log_level="warning") 
    server = uvicorn.Server(config)
    server.run()

t = threading.Thread(target=_run, daemon=True)
t.start()

import time; time.sleep(1) # Wait for startup

print(f"✅ TTS server running on http://127.0.0.1:{PORT}")
print(f"   Voice: {VOICE}")

# ── Auto-Config: Update my-app ────────────────────────────────────────────────
try:
    cwd = os.getcwd()
    print(f"   Current Dir: {cwd}")
    
    # Locate my-app (Robustness for VSCode vs Colab vs subdirs)
    my_app_path = None
    possible_paths = [
        os.path.join(cwd, "my-app"),
        os.path.join(os.path.dirname(cwd), "my-app"),
        os.path.abspath("my-app"),
        r"e:\coding\interview\my-app" # User's specific path fallback
    ]
    
    for p in possible_paths:
        if os.path.exists(p) and os.path.isdir(p):
            my_app_path = p
            break
            
    if my_app_path:
        # 1. Write Port File (Primary Sync Method)
        port_file = os.path.join(my_app_path, "tts_port.txt")
        with open(port_file, "w") as f:
            f.write(str(PORT))
        print(f"✅ Configured: {port_file}")
        
        # 2. Update .env (Secondary/Docs)
        env_file = os.path.join(my_app_path, ".env")
        if os.path.exists(env_file):
            with open(env_file, "r", encoding="utf-8") as f: content = f.read()
            
            new_url = f"http://127.0.0.1:{PORT}/v1/tts"
            if "DIA_TTS_API_URL=" in content:
                content = re.sub(r"DIA_TTS_API_URL=.*", f"DIA_TTS_API_URL={new_url}", content)
            else:
                if content and not content.endswith("\n"): content += "\n"
                content += f"DIA_TTS_API_URL={new_url}\n"
                
            with open(env_file, "w", encoding="utf-8") as f: f.write(content)
            print(f"✅ Updated .env: DIA_TTS_API_URL={new_url}")
    else:
        print("⚠️ 'my-app' folder not found automatically. (Skipping file updates)")
        print(f"   Please update .env manually to port {PORT}")

except Exception as e:
    print(f"⚠️ Config update warning: {e}")
    # Do not raise, server is still good!

print("\n" + "="*60)
print(f"👉 Server is ready! Restart Next.js if you haven't.")
print("="*60 + "\n")

✅ TTS server running on http://127.0.0.1:8081
   Voice: en-US-JennyNeural
   Current Dir: /content
⚠️ 'my-app' folder not found automatically. (Skipping file updates)
   Please update .env manually to port 8081

👉 Server is ready! Restart Next.js if you haven't.



In [3]:

# ── Cell 6: Expose via ngrok (public URL) ─────────────────────────────────────
# Sign up free at https://ngrok.com and paste your token below.
import sys, subprocess, secrets, requests

def find_server_port(start=8080, max_attempts=10):
    """Auto-detect which port the TTS server is running on."""
    for port in range(start, start + max_attempts):
        try:
            resp = requests.get(f"http://localhost:{port}/health", timeout=1)
            if resp.status_code == 200:
                return port
        except:
            pass
    return None

# Find the actual server port (Cell 5 may have chosen 8080, 8081, etc.)
PORT = find_server_port(8080)
if PORT is None:
    print("❌ Could not find Dia TTS server on ports 8080-8090")
    print("   Make sure Cell 5 is still running")
    raise RuntimeError("Server not found for ngrok tunnel")

print(f"✅ Found Dia TTS server on port {PORT}")

API_KEY = secrets.token_urlsafe(24)  # Generate a new key for this session

# Ensure pyngrok is installed
try:
    from pyngrok import ngrok, conf
except ImportError:
    print("📦 Installing pyngrok...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyngrok", "-q"])
    from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = "3AsMkVflELYbGl5Mcyi5dEb9qOU_6og7ioE6qhhPaRUVLrj7D"   # ← replace this

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any leftover tunnels
ngrok.kill()

tunnel = ngrok.connect(PORT, "http")

PUBLIC_URL = tunnel.public_url

print("=" * 60)
print(f"  Dia TTS Public URL  : {PUBLIC_URL}")
print("=" * 60)
print()
print("Add these to my-app/.env.local  (or Vercel / Railway env):")
print()
print(f"  DIA_TTS_API_URL={PUBLIC_URL}/v1/tts")
print(f"  DIA_TTS_API_KEY={API_KEY}")
print()
print("Then restart the Next.js dev server:  npm run dev")


✅ Found Dia TTS server on port 8081
  Dia TTS Public URL  : https://multiradial-davida-presagefully.ngrok-free.dev

Add these to my-app/.env.local  (or Vercel / Railway env):

  DIA_TTS_API_URL=https://multiradial-davida-presagefully.ngrok-free.dev/v1/tts
  DIA_TTS_API_KEY=EBn_Cz3rSRWAyPhTzeeWSbkkUW_fb-FG

Then restart the Next.js dev server:  npm run dev


In [ ]:
# ── Cell 7: Test Connection & Save to temp_voice ──────────────────────────────
import requests
import time
import os
from IPython.display import Audio, display

# 1. Setup Output Directory
OUTPUT_DIR = "temp_voice"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Config - Auto-discover port
def find_server_port(start=8080, max_attempts=10):
    for port in range(start, start + max_attempts):
        try:
            resp = requests.get(f"http://localhost:{port}/health", timeout=0.5)
            if resp.status_code == 200:
                return port
        except:
            pass
    return None

found_port = find_server_port()
PORT = found_port if found_port else 8080

BASE_URL = f"http://localhost:{PORT}"
TIMEOUT = 10
SAMPLES = ["Hi, this is a test saved to the temp voice folder.", "The audio is crystal clear."]

print(f"Testing on {BASE_URL}...")

if not found_port:
    print("⚠️ Warning: Could not auto-detect running server. Is Cell 5 running?")

for i, text in enumerate(SAMPLES, 1):
    try:
        resp = requests.post(f"{BASE_URL}/v1/tts", json={"text": text}, timeout=TIMEOUT)
        if resp.status_code == 200:
            filename = f"sample_{i}.mp3"
            path = os.path.join(OUTPUT_DIR, filename)
            
            with open(path, "wb") as f:
                f.write(resp.content)
            
            print(f"✅ Saved: {path} ({len(resp.content)/1024:.1f} KB)")
            display(Audio(path))
        else:
            print(f"❌ HTTP {resp.status_code}: {resp.text}")
    except Exception as e:
        print(f"❌ Error: {e}")

Testing on http://localhost:8081...
✅ Saved: temp_voice/sample_1.mp3 (23.5 KB)


✅ Saved: temp_voice/sample_2.mp3 (15.3 KB)
